In [ ]:
# CODE 1
import cv2

cap = cv2.VideoCapture(0)

tracker = cv2.TrackerCSRT_create()
initBB = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if initBB is not None:
        success, box = tracker.update(frame)
        if success:
            (x, y, w, h) = [int(v) for v in box]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        else:
            cv2.putText(frame, "Tracking lost!", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Face Tracking (Live)", frame)

    key = cv2.waitKey(30) & 0xFF
    if key == ord("s"): 
        initBB = cv2.selectROI("Face Tracking (Live)", frame, fromCenter=False, showCrosshair=True)
        tracker.init(frame, initBB)
    elif key == 27:  
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
# CODE 2
import cv2
import mediapipe as mp

# Mediapipe 
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

#webcam
cap = cv2.VideoCapture(0)

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        #RGB
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_detection.process(img_rgb)

        
        if results.detections:
            for det in results.detections:
                mp_drawing.draw_detection(frame, det)

        cv2.imshow("Face Detection (Live)", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC
            break

cap.release()
cv2.destroyAllWindows()


In [ ]:
# CODE 3
import cv2
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=2,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(img_rgb)

        if results.multi_face_landmarks:
            for landmarks in results.multi_face_landmarks:
                mp_drawing.draw_landmarks(
                    frame, landmarks, mp_face_mesh.FACEMESH_TESSELATION,
                    mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1),
                    mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=1, circle_radius=1)
                )

        cv2.imshow("Face Landmark Detection (Live)", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC
            break

cap.release()
cv2.destroyAllWindows()


In [ ]:
# CODE 4
import face_recognition
import cv2
import os
import pickle
import numpy as np

KNOWN_FACES_DIR = "known_faces"
EMBEDDINGS_FILE = "embeddings.pkl"

known_embeddings = []
known_names = []

# Allowed image extensions
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

for name in os.listdir(KNOWN_FACES_DIR):
    person_dir = os.path.join(KNOWN_FACES_DIR, name)
    if not os.path.isdir(person_dir):
        continue

    for filename in os.listdir(person_dir):
        if not filename.lower().endswith(VALID_EXTS):
            print(f"⚠️ Skipping unsupported file: {filename}")
            continue

        file_path = os.path.join(person_dir, filename)
        try:
            # Load and ensure correct format
            img = face_recognition.load_image_file(file_path)
            if img.dtype != np.uint8:
                img = img.astype(np.uint8)

            # Get face encodings
            encodings = face_recognition.face_encodings(img)
            if len(encodings) > 0:
                known_embeddings.append(encodings[0])
                known_names.append(name)
                print(f"✅ Encoded {filename} for {name}")
            else:
                print(f"❌ No face detected in {filename}")

        except Exception as e:
            print(f"🚫 Error processing {filename}: {e}")

# Save embeddings
with open(EMBEDDINGS_FILE, "wb") as f:
    pickle.dump((known_embeddings, known_names), f)

print("\n✅ Face Embeddings Saved Successfully")
print(f"Total faces encoded: {len(known_embeddings)}")


ModuleNotFoundError: No module named 'face_recognition'

In [ ]:
# CODE 5
import cv2
import face_recognition
import pickle

EMBEDDINGS_FILE = "embeddings.pkl"
known_embeddings, known_names = pickle.load(open(EMBEDDINGS_FILE, "rb"))

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    locations = face_recognition.face_locations(rgb)
    encodings = face_recognition.face_encodings(rgb, locations)

    for enc, (top, right, bottom, left) in zip(encodings, locations):
        matches = face_recognition.compare_faces(known_embeddings, enc, tolerance=0.5)
        name = "Unknown"

        if True in matches:
            index = matches.index(True)
            name = known_names[index]

        cv2.rectangle(frame, (left, top), (right, bottom), (0,255,0), 2)
        cv2.putText(frame, name, (left, top-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    cv2.imshow("Face Recognition - Live", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


RuntimeError: Unsupported image type, must be 8bit gray or RGB image.

In [ ]:
# CODE 5
import cv2
import mediapipe as mp
from deepface import DeepFace
from collections import deque, Counter

 
# Initialize MediaPipe Detector
 
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.6)

 
# Webcam Capture
 
cap = cv2.VideoCapture(0)

 
# Frame control (for speed)
 
frame_count = 0
skip_frames = 5   # Run emotion model every 5 frames

 
# Emotion smoothing buffer

emotion_buffer = deque(maxlen=10)

print("Press 'q' to exit")

while True:

    ret, frame = cap.read()
    if not ret:
        break

    # Convert BGR → RGB (correct conversion)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

     
    # Face Detection
    
    results = face_detector.process(rgb_frame)

    if results.detections:

        for detection in results.detections:

            bbox = detection.location_data.relative_bounding_box
            h, w, _ = frame.shape

            x = int(bbox.xmin * w)
            y = int(bbox.ymin * h)
            bw = int(bbox.width * w)
            bh = int(bbox.height * h)

            # Keep coordinates valid
            x = max(0, x)
            y = max(0, y)

            face_roi = rgb_frame[y:y+bh, x:x+bw]

            emotion = "Detecting..."

             
            # Run emotion model occasionally
             
            if frame_count % skip_frames == 0:

                try:
                    result = DeepFace.analyze(
                        face_roi,
                        actions=['emotion'],
                        enforce_detection=False
                    )

                    emotion = result[0]['dominant_emotion']

                    # Add emotion to smoothing buffer
                    emotion_buffer.append(emotion)

                except:
                    emotion = "Unknown"

         
            # Smooth prediction
             
            if len(emotion_buffer) > 0:
                emotion = Counter(emotion_buffer).most_common(1)[0][0]

             
            # Draw bounding box
         
            cv2.rectangle(frame, (x, y), (x+bw, y+bh), (0,255,0), 2)

            cv2.putText(
                frame,
                emotion,
                (x, y1),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0,255,0),
                2
            )

     
    # Display Frame
     
    cv2.imshow("RealTime Emotion Detection", frame)

    frame_count += 1

    # Exit key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


 
# Cleanup

cap.release()
cv2.destroyAllWindows()

Press 'q' to exit


NameError: name 'y1' is not defined

In [ ]:
# CODE 6
import cv2
import mediapipe as mp
from deepface import DeepFace

print("All libraries loaded successfully")


All libraries loaded successfully


In [ ]:
# CODE 7
import cv2
print(cv2.__file__)


c:\Users\abhis\OneDrive\Desktop\final_year_project\my_code\venv310\lib\site-packages\cv2\__init__.py


In [ ]:
# CODE 8
import cv2
print(cv2)
print(dir(cv2))

<module 'cv2' from 'c:\\Users\\abhis\\OneDrive\\Desktop\\final_year_project\\my_code\\venv310\\lib\\site-packages\\cv2\\__init__.py'>
['ACCESS_FAST', 'ACCESS_MASK', 'ACCESS_READ', 'ACCESS_RW', 'ACCESS_WRITE', 'ADAPTIVE_THRESH_GAUSSIAN_C', 'ADAPTIVE_THRESH_MEAN_C', 'AGAST_FEATURE_DETECTOR_AGAST_5_8', 'AGAST_FEATURE_DETECTOR_AGAST_7_12D', 'AGAST_FEATURE_DETECTOR_AGAST_7_12S', 'AGAST_FEATURE_DETECTOR_NONMAX_SUPPRESSION', 'AGAST_FEATURE_DETECTOR_OAST_9_16', 'AGAST_FEATURE_DETECTOR_THRESHOLD', 'AKAZE', 'AKAZE_DESCRIPTOR_KAZE', 'AKAZE_DESCRIPTOR_KAZE_UPRIGHT', 'AKAZE_DESCRIPTOR_MLDB', 'AKAZE_DESCRIPTOR_MLDB_UPRIGHT', 'AKAZE_create', 'ALGO_HINT_ACCURATE', 'ALGO_HINT_APPROX', 'ALGO_HINT_DEFAULT', 'AffineFeature', 'AffineFeature_create', 'AffineTransformer', 'AgastFeatureDetector', 'AgastFeatureDetector_AGAST_5_8', 'AgastFeatureDetector_AGAST_7_12d', 'AgastFeatureDetector_AGAST_7_12s', 'AgastFeatureDetector_NONMAX_SUPPRESSION', 'AgastFeatureDetector_OAST_9_16', 'AgastFeatureDetector_THRESHOLD',

In [ ]:
# CODE 9
# emotion classification base 1 ONE

import cv2
import mediapipe as mp
from deepface import DeepFace
import time

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

prev_time = 0

with mp_face_detection.FaceDetection(
        model_selection=0,
        min_detection_confidence=0.5) as face_detection:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_detection.process(rgb_frame)

        if results.detections:
            for detection in results.detections:

                bbox = detection.location_data.relative_bounding_box
                h, w, _ = frame.shape

                x = int(bbox.xmin * w)
                y = int(bbox.ymin * h)
                width = int(bbox.width * w)
                height = int(bbox.height * h)

                face = frame[y:y+height, x:x+width]

                try:
                    analysis = DeepFace.analyze(
                        face,
                        actions=['emotion'],
                        enforce_detection=False
                    )

                    emotion = analysis[0]['dominant_emotion']

                    cv2.putText(frame, emotion,
                                (x, y-10),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.9,
                                (0,255,0),
                                2)

                except:
                    pass

                cv2.rectangle(frame,
                              (x, y),
                              (x+width, y+height),
                              (0,255,0),
                              2)

        # FPS counter
        curr_time = time.time()
        fps = 1/(curr_time-prev_time) if curr_time-prev_time != 0 else 0
        prev_time = curr_time

        cv2.putText(frame, f"FPS: {int(fps)}",
                    (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)

        cv2.imshow("Emotion Detection", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# CODE 10
#Emotion classifier base 2 around max 9 fps

import cv2
import mediapipe as mp
from deepface import DeepFace
import time
import numpy as np
from collections import deque

mp_face_detection = mp.solutions.face_detection
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

emotion_buffer = deque(maxlen=10)

prev_time = 0

with mp_face_detection.FaceDetection(min_detection_confidence=0.6) as face_detection, \
     mp_face_mesh.FaceMesh(static_image_mode=False,
                           max_num_faces=1,
                           refine_landmarks=True) as face_mesh:

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = face_detection.process(rgb)

        if results.detections:

            for detection in results.detections:

                bbox = detection.location_data.relative_bounding_box
                h, w, _ = frame.shape

                x = int(bbox.xmin * w)
                y = int(bbox.ymin * h)
                width = int(bbox.width * w)
                height = int(bbox.height * h)

                face = frame[y:y+height, x:x+width]

                if face.size == 0:
                    continue

                face = cv2.resize(face, (224,224))

                try:

                    analysis = DeepFace.analyze(
                        face,
                        actions=['emotion'],
                        enforce_detection=False
                    )

                    emotion = analysis[0]['dominant_emotion']

                    emotion_buffer.append(emotion)

                    # emotion smoothing
                    final_emotion = max(set(emotion_buffer),
                                        key=emotion_buffer.count)

                    cv2.putText(frame,
                                final_emotion,
                                (x, y-10),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                1,
                                (0,255,0),
                                2)

                except:
                    pass

                cv2.rectangle(frame,
                              (x,y),
                              (x+width,y+height),
                              (0,255,0),
                              2)

        # FPS counter
        curr_time = time.time()
        fps = 1/(curr_time-prev_time) if curr_time-prev_time!=0 else 0
        prev_time = curr_time

        cv2.putText(frame,
                    f"FPS: {int(fps)}",
                    (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)

        cv2.imshow("Emotion Detection", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# CODE 11
#emotion classifier base 3 ~25-30 fps ###### PREFERED ##########

import cv2
import mediapipe as mp
from deepface import DeepFace
import time
import numpy as np
from collections import deque

mp_face_detection = mp.solutions.face_detection
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

emotion_history = deque(maxlen=15)

frame_count = 0
prev_time = 0

with mp_face_detection.FaceDetection(min_detection_confidence=0.7) as face_detection, \
     mp_face_mesh.FaceMesh(static_image_mode=False,
                           max_num_faces=1,
                           refine_landmarks=True,
                           min_detection_confidence=0.6,
                           min_tracking_confidence=0.6) as face_mesh:

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        detection_results = face_detection.process(rgb)
        mesh_results = face_mesh.process(rgb)

        if detection_results.detections:

            for detection in detection_results.detections:

                bbox = detection.location_data.relative_bounding_box

                x = max(0, int(bbox.xmin * w))
                y = max(0, int(bbox.ymin * h))
                width = int(bbox.width * w)
                height = int(bbox.height * h)

                x2 = min(w, x + width)
                y2 = min(h, y + height)

                face = frame[y:y2, x:x2]

                if face.size == 0:
                    continue

                # Improve crop using facemesh landmarks
                if mesh_results.multi_face_landmarks:

                    landmarks = mesh_results.multi_face_landmarks[0]

                    xs = [lm.x for lm in landmarks.landmark]
                    ys = [lm.y for lm in landmarks.landmark]

                    xmin = int(min(xs) * w)
                    xmax = int(max(xs) * w)
                    ymin = int(min(ys) * h)
                    ymax = int(max(ys) * h)

                    xmin = max(0, xmin)
                    ymin = max(0, ymin)
                    xmax = min(w, xmax)
                    ymax = min(h, ymax)

                    face = frame[ymin:ymax, xmin:xmax]

                face = cv2.resize(face, (224,224))

                frame_count += 1

                # Run DeepFace every 5 frames (faster + more stable)
                if frame_count % 5 == 0:

                    try:
                        analysis = DeepFace.analyze(
                            face,
                            actions=['emotion'],
                            enforce_detection=False
                        )

                        emotions = analysis[0]['emotion']

                        emotion_history.append(emotions)

                    except:
                        pass

                # Smooth predictions using averaged probabilities
                if emotion_history:

                    avg_emotions = {}

                    for e in emotion_history:
                        for k,v in e.items():
                            avg_emotions[k] = avg_emotions.get(k,0) + v

                    for k in avg_emotions:
                        avg_emotions[k] /= len(emotion_history)

                    final_emotion = max(avg_emotions, key=avg_emotions.get)

                    cv2.putText(frame,
                                final_emotion,
                                (x, y-10),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                1,
                                (0,255,0),
                                2)

                cv2.rectangle(frame,(x,y),(x2,y2),(0,255,0),2)

        # FPS calculation
        curr_time = time.time()
        fps = 1/(curr_time-prev_time) if curr_time-prev_time!=0 else 0
        prev_time = curr_time

        cv2.putText(frame,
                    f"FPS: {int(fps)}",
                    (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)

        cv2.imshow("Enhanced Emotion Detection", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# CODE 12
"""

expression feature engineering base 1 

Real-Time Facial Emotion Detection System
Architecture: Webcam → MediaPipe FaceDetection → FaceMesh → DeepFace → Temporal Smoothing
claude

"""

import cv2
import numpy as np
import mediapipe as mp
from deepface import DeepFace
from collections import deque
import time


 
# Configuration Constants

FRAME_SKIP          = 5       # Run DeepFace every N frames
SMOOTH_BUFFER_SIZE  = 8       # Number of frames for temporal smoothing
CROP_PADDING        = 0.20    # Fractional padding around landmark bounding box
FACE_CROP_SIZE      = 224     # DeepFace input resolution
MIN_DETECTION_CONF  = 0.6     # MediaPipe face detection confidence threshold
MIN_TRACKING_CONF   = 0.6     # MediaPipe face mesh tracking confidence threshold

EMOTION_LABELS = [
    "angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"
]

# Display colours (BGR)
BOX_COLOR       = (0, 220, 110)
TEXT_COLOR      = (255, 255, 255)
LABEL_BG_COLOR  = (0, 160, 80)
FPS_COLOR       = (0, 200, 255)


 
# Helper: Detect primary face with MediaPipe

def detect_face(frame, face_detector):
    """
    Use MediaPipe FaceDetection to locate the largest face in the frame.

    Returns:
        (x1, y1, x2, y2) bounding box in pixel coords, or None if no face found.
    """
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_detector.process(rgb)

    if not results.detections:
        return None

    # Pick the detection with the highest confidence score
    best = max(results.detections, key=lambda d: d.score[0])

    bb = best.location_data.relative_bounding_box
    x1 = int(bb.xmin * w)
    y1 = int(bb.ymin * h)
    x2 = int((bb.xmin + bb.width) * w)
    y2 = int((bb.ymin + bb.height) * h)

    # Clamp to frame boundaries
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w - 1, x2), min(h - 1, y2)

    if x2 <= x1 or y2 <= y1:
        return None

    return (x1, y1, x2, y2)



# Helper: Refine crop using FaceMesh landmarks
 
def refine_crop_with_facemesh(frame, coarse_box, face_mesh):
    """
    Run MediaPipe FaceMesh on the coarse crop to obtain precise landmark bounds.
    Adds proportional padding and returns the refined (x1,y1,x2,y2) in full-frame
    coordinates, or falls back to the coarse box if no mesh is found.

    Returns:
        (x1, y1, x2, y2) refined bounding box, or None on invalid region.
    """
    h, w = frame.shape[:2]
    cx1, cy1, cx2, cy2 = coarse_box

    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None

    rgb_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    results  = face_mesh.process(rgb_crop)

    ch, cw = crop.shape[:2]

    if results.multi_face_landmarks:
        lm = results.multi_face_landmarks[0].landmark
        xs = [p.x * cw for p in lm]
        ys = [p.y * ch for p in lm]

        lx1 = int(min(xs)); ly1 = int(min(ys))
        lx2 = int(max(xs)); ly2 = int(max(ys))

        # Convert back to full-frame coordinates
        lx1 += cx1; ly1 += cy1
        lx2 += cx1; ly2 += cy1
    else:
        # Fall back to coarse detection box
        lx1, ly1, lx2, ly2 = cx1, cy1, cx2, cy2

    # Apply padding
    pad_x = int((lx2 - lx1) * CROP_PADDING)
    pad_y = int((ly2 - ly1) * CROP_PADDING)

    rx1 = max(0,     lx1 - pad_x)
    ry1 = max(0,     ly1 - pad_y)
    rx2 = min(w - 1, lx2 + pad_x)
    ry2 = min(h - 1, ly2 + pad_y)

    if rx2 <= rx1 or ry2 <= ry1:
        return None

    return (rx1, ry1, rx2, ry2)



# Helper: Analyse emotion with DeepFace

def analyze_emotion(frame, box):
    """
    Crop the face region, resize to FACE_CROP_SIZE, and run DeepFace emotion
    analysis.

    Returns:
        dict mapping emotion label → probability (0–100), or None on failure.
    """
    x1, y1, x2, y2 = box
    face_crop = frame[y1:y2, x1:x2]

    if face_crop.size == 0:
        return None

    # Resize to standard input expected by DeepFace / FER model
    face_resized = cv2.resize(face_crop, (FACE_CROP_SIZE, FACE_CROP_SIZE))

    try:
        result = DeepFace.analyze(
            img_path       = face_resized,
            actions        = ["emotion"],
            enforce_detection = False,   # already cropped — skip internal detector
            silent         = True
        )
        # DeepFace may return a list or a single dict
        if isinstance(result, list):
            result = result[0]

        return result.get("emotion", None)

    except Exception:
        return None



# Helper: Temporal smoothing of emotion scores
 
def smooth_emotion(emotion_buffer, new_scores):
    """
    Push the latest per-emotion probability dict into a rolling buffer and
    return the smoothed dominant emotion label.

    Args:
        emotion_buffer: deque of recent emotion-score dicts
        new_scores:     dict {label: score} from DeepFace (or None)

    Returns:
        dominant emotion label (str), or "unknown"
    """
    if new_scores is None:
        if not emotion_buffer:
            return "unknown"
        # Reuse last buffer entry unchanged
        new_scores = emotion_buffer[-1]

    emotion_buffer.append(new_scores)

    # Average each emotion's score across the buffer
    avg_scores = {label: 0.0 for label in EMOTION_LABELS}
    for scores in emotion_buffer:
        for label in EMOTION_LABELS:
            avg_scores[label] += scores.get(label, 0.0)

    count = len(emotion_buffer)
    avg_scores = {k: v / count for k, v in avg_scores.items()}

    return max(avg_scores, key=avg_scores.get)


# 
# Drawing utilities
# 
def draw_overlay(frame, box, emotion, fps):
    """Draw bounding box, emotion label badge, and FPS counter on the frame."""
    x1, y1, x2, y2 = box

    # Bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), BOX_COLOR, 2)

    # Emotion badge
    label     = f"  {emotion.upper()}  "
    font      = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.75
    thickness  = 2
    (tw, th), baseline = cv2.getTextSize(label, font, font_scale, thickness)

    badge_y1 = max(0, y1 - th - baseline - 8)
    badge_y2 = y1
    badge_x2 = min(frame.shape[1], x1 + tw + 4)

    cv2.rectangle(frame, (x1, badge_y1), (badge_x2, badge_y2), LABEL_BG_COLOR, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - baseline - 2),
                font, font_scale, TEXT_COLOR, thickness, cv2.LINE_AA)

    # FPS counter (top-left corner)
    cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, FPS_COLOR, 2, cv2.LINE_AA)


# 
# Main
# 
def main():
    #  Initialise MediaPipe modules 
    mp_face_detection = mp.solutions.face_detection
    mp_face_mesh      = mp.solutions.face_mesh

    face_detector = mp_face_detection.FaceDetection(
        model_selection    = 0,                  # 0 = short-range (≤2 m), ideal for webcam
        min_detection_confidence = MIN_DETECTION_CONF
    )
    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode        = False,
        max_num_faces            = 1,
        refine_landmarks         = True,
        min_detection_confidence = MIN_DETECTION_CONF,
        min_tracking_confidence  = MIN_TRACKING_CONF
    )

    #  Temporal smoothing buffer ─
    emotion_buffer = deque(maxlen=SMOOTH_BUFFER_SIZE)

    #  State variables ─
    frame_count       = 0
    last_emotion      = "unknown"
    last_scores       = None
    last_box          = None

    # FPS tracking
    fps          = 0.0
    fps_timer    = time.time()
    fps_frames   = 0

    #  Open webcam ─
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        return

    # Lower internal buffer to reduce latency
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    print("[INFO] Starting emotion detection. Press 'q' to quit.")

    #  Main loop ─
    while True:
        ret, frame = cap.read()
        if not ret:
            print("[WARNING] Failed to read frame. Retrying…")
            continue

        frame_count += 1
        fps_frames  += 1

        # Update FPS every second
        elapsed = time.time() - fps_timer
        if elapsed >= 1.0:
            fps        = fps_frames / elapsed
            fps_timer  = time.time()
            fps_frames = 0

        #  Step 1: Fast face detection 
        coarse_box = detect_face(frame, face_detector)

        if coarse_box is not None:
            #  Step 2: Refine crop with FaceMesh ─
            refined_box = refine_crop_with_facemesh(frame, coarse_box, face_mesh)
            if refined_box is None:
                refined_box = coarse_box  # graceful fallback

            last_box = refined_box

            #  Step 3: Run DeepFace every FRAME_SKIP frames 
            if frame_count % FRAME_SKIP == 0:
                last_scores = analyze_emotion(frame, refined_box)

            #  Step 4: Temporal smoothing 
            last_emotion = smooth_emotion(emotion_buffer, last_scores)

        #  Step 5: Draw results (only when a face was previously found) 
        if last_box is not None and last_emotion != "unknown":
            draw_overlay(frame, last_box, last_emotion, fps)
        else:
            # Still show FPS even without a face
            cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, FPS_COLOR, 2, cv2.LINE_AA)
            cv2.putText(frame, "No face detected", (12, 68),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (80, 80, 255), 2, cv2.LINE_AA)

        cv2.imshow("Emotion Detection  [q = quit]", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    #  Cleanup 
    cap.release()
    face_detector.close()
    face_mesh.close()
    cv2.destroyAllWindows()
    print("[INFO] Stopped.")


if __name__ == "__main__":
    main()

[INFO] Starting emotion detection. Press 'q' to quit.
[INFO] Stopped.


In [ ]:
 # CODE 13
# Emotion feature engg base 2 with score 

import cv2
import mediapipe as mp
from deepface import DeepFace
import time
import numpy as np
from collections import deque
import csv
from datetime import datetime

mp_face_detection = mp.solutions.face_detection
mp_face_mesh = mp.solutions.face_mesh

cap = cv2.VideoCapture(0)

emotion_history = deque(maxlen=15)

frame_count = 0
prev_time = 0

# Emotion counters for session analysis
emotion_counts = {
    "happy":0,
    "neutral":0,
    "sad":0,
    "angry":0,
    "surprise":0,
    "fear":0,
    "disgust":0
}

# Create log file
log_file = open("emotion_log.csv","a",newline="")
csv_writer = csv.writer(log_file)

csv_writer.writerow(["timestamp","emotion","confidence"])

with mp_face_detection.FaceDetection(min_detection_confidence=0.7) as face_detection, \
     mp_face_mesh.FaceMesh(static_image_mode=False,
                           max_num_faces=1,
                           refine_landmarks=True) as face_mesh:

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        detection_results = face_detection.process(rgb)
        mesh_results = face_mesh.process(rgb)

        if detection_results.detections:

            for detection in detection_results.detections:

                bbox = detection.location_data.relative_bounding_box

                x = max(0, int(bbox.xmin * w))
                y = max(0, int(bbox.ymin * h))
                width = int(bbox.width * w)
                height = int(bbox.height * h)

                x2 = min(w, x + width)
                y2 = min(h, y + height)

                face = frame[y:y2, x:x2]

                if face.size == 0:
                    continue

                face = cv2.resize(face,(224,224))

                frame_count += 1

                if frame_count % 5 == 0:

                    try:
                        analysis = DeepFace.analyze(
                            face,
                            actions=['emotion'],
                            enforce_detection=False
                        )

                        emotions = analysis[0]['emotion']

                        emotion_history.append(emotions)

                    except:
                        pass

                if emotion_history:

                    avg_emotions = {}

                    for e in emotion_history:
                        for k,v in e.items():
                            avg_emotions[k] = avg_emotions.get(k,0) + v

                    for k in avg_emotions:
                        avg_emotions[k] /= len(emotion_history)

                    final_emotion = max(avg_emotions,key=avg_emotions.get)
                    confidence = avg_emotions[final_emotion]

                    # Log emotion
                    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    csv_writer.writerow([timestamp,final_emotion,confidence])

                    # Update session statistics
                    if final_emotion in emotion_counts:
                        emotion_counts[final_emotion]+=1

                    cv2.putText(frame,
                                f"{final_emotion} ({confidence:.2f})",
                                (x,y-10),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.9,
                                (0,255,0),
                                2)

                cv2.rectangle(frame,(x,y),(x2,y2),(0,255,0),2)

        # FPS counter
        curr_time = time.time()
        fps = 1/(curr_time-prev_time) if curr_time-prev_time!=0 else 0
        prev_time = curr_time

        cv2.putText(frame,
                    f"FPS: {int(fps)}",
                    (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)

        cv2.imshow("Emotion Detection + Logging",frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()
log_file.close()

# Mental health score (simple heuristic)
positive = emotion_counts["happy"]
neutral = emotion_counts["neutral"]
negative = emotion_counts["sad"] + emotion_counts["angry"] + emotion_counts["fear"] + emotion_counts["disgust"]

total = sum(emotion_counts.values())

if total>0:
    mental_score = ((positive + 0.5*neutral)/total)*100
else:
    mental_score = 50

print("\nSession Emotion Statistics")
print(emotion_counts)

print(f"\nMental Health Score: {mental_score:.2f}/100")


Session Emotion Statistics
{'happy': 0, 'neutral': 131, 'sad': 0, 'angry': 0, 'surprise': 0, 'fear': 0, 'disgust': 0}

Mental Health Score: 50.00/100


In [ ]:
# CODE 14
"""

emotion aggregator base 1 with log below execution cell with output 

Real-Time Facial Emotion Detection System — Enhanced
=====================================================
Pipeline : Webcam → MediaPipe FaceDetection → FaceMesh crop →
           DeepFace (every 5 frames) → Temporal smoothing → Display
Extras   : CSV logging  |  Session statistics  |  Mental-health score
"""

import csv
import os
import time
from collections import deque, defaultdict
from datetime import datetime

import cv2
import mediapipe as mp
import numpy as np
from deepface import DeepFace


# 
# Configuration
# 
FRAME_SKIP         = 5       # DeepFace runs once every N frames
SMOOTH_BUFFER_SIZE = 8       # Rolling window for temporal smoothing
CROP_PADDING       = 0.20    # Fractional padding around landmark bbox
FACE_CROP_SIZE     = 224     # Input size expected by DeepFace / FER
MIN_DETECT_CONF    = 0.60
MIN_TRACK_CONF     = 0.60

LOG_DIR            = "emotion_logs"
CSV_FILENAME       = os.path.join(
    LOG_DIR, f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
)

EMOTION_LABELS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]

# Valence weights used for the mental-health score  (+1 positive, −1 negative)
VALENCE = {
    "happy":    1.0,
    "neutral":  0.3,
    "surprise": 0.1,
    "sad":     -0.6,
    "angry":   -0.8,
    "fear":    -0.7,
    "disgust": -0.5,
}

#  Display colours (BGR) 
BOX_COLOR      = (0, 220, 110)
TEXT_COLOR     = (255, 255, 255)
BADGE_COLOR    = (0, 160, 80)
FPS_COLOR      = (0, 200, 255)
CONF_COLOR     = (220, 220,  60)


# 
# CSV / Logging helpers
# 
def init_csv(path: str) -> csv.writer:
    """Create (or append to) the log CSV and return an open writer."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fh = open(path, "a", newline="", encoding="utf-8")
    writer = csv.writer(fh)
    # Write header only for new files
    if os.path.getsize(path) == 0:
        writer.writerow(["timestamp", "emotion", "confidence_pct"])
    return writer, fh


def log_emotion(writer, emotion: str, confidence: float):
    """Append one row to the CSV log."""
    writer.writerow([
        datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        emotion,
        f"{confidence:.2f}",
    ])



# Face detection  (MediaPipe FaceDetection)

def detect_face(frame, detector):
    """
    Locate the highest-confidence face in *frame*.

    Returns (x1, y1, x2, y2) pixel bbox, or None when no face is found.
    """
    h, w = frame.shape[:2]
    results = detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if not results.detections:
        return None

    best = max(results.detections, key=lambda d: d.score[0])
    bb   = best.location_data.relative_bounding_box

    x1 = max(0,     int(bb.xmin * w))
    y1 = max(0,     int(bb.ymin * h))
    x2 = min(w - 1, int((bb.xmin + bb.width)  * w))
    y2 = min(h - 1, int((bb.ymin + bb.height) * h))

    return (x1, y1, x2, y2) if x2 > x1 and y2 > y1 else None


# 
# Landmark-based crop refinement  (MediaPipe FaceMesh)
# 
def refine_crop_with_facemesh(frame, coarse_box, face_mesh):
    """
    Run FaceMesh on the coarse crop to tighten the bounding box around
    the actual facial landmarks, then re-pad.

    Returns refined (x1, y1, x2, y2) in full-frame coords,
    or falls back to *coarse_box* when the mesh finds nothing.
    """
    h, w = frame.shape[:2]
    cx1, cy1, cx2, cy2 = coarse_box

    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None

    ch, cw = crop.shape[:2]
    results = face_mesh.process(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

    if results.multi_face_landmarks:
        pts = results.multi_face_landmarks[0].landmark
        xs  = [p.x * cw for p in pts]
        ys  = [p.y * ch for p in pts]
        # Convert landmark bbox back to full-frame space
        lx1 = int(min(xs)) + cx1;  ly1 = int(min(ys)) + cy1
        lx2 = int(max(xs)) + cx1;  ly2 = int(max(ys)) + cy1
    else:
        lx1, ly1, lx2, ly2 = cx1, cy1, cx2, cy2

    # Add proportional padding
    px = int((lx2 - lx1) * CROP_PADDING)
    py = int((ly2 - ly1) * CROP_PADDING)

    rx1 = max(0,     lx1 - px);  ry1 = max(0,     ly1 - py)
    rx2 = min(w - 1, lx2 + px);  ry2 = min(h - 1, ly2 + py)

    return (rx1, ry1, rx2, ry2) if rx2 > rx1 and ry2 > ry1 else None


# 
# DeepFace emotion analysis
# 
def analyze_emotion(frame, box):
    """
    Crop + resize the face region to FACE_CROP_SIZE and call DeepFace.

    Returns (scores_dict, dominant_emotion, confidence_pct)
            where scores_dict maps label → probability (0–100).
    Returns (None, None, None) on any failure.
    """
    x1, y1, x2, y2 = box
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None, None, None

    resized = cv2.resize(crop, (FACE_CROP_SIZE, FACE_CROP_SIZE))

    try:
        result = DeepFace.analyze(
            img_path          = resized,
            actions           = ["emotion"],
            enforce_detection = False,   # face is already isolated
            silent            = True,
        )
        if isinstance(result, list):
            result = result[0]

        scores    = result.get("emotion", {})
        dominant  = result.get("dominant_emotion", max(scores, key=scores.get))
        confidence = scores.get(dominant, 0.0)
        return scores, dominant, confidence

    except Exception:
        return None, None, None



# Temporal smoothing
# 
def smooth_emotion(buffer: deque, new_scores: dict):
    """
    Push *new_scores* into the rolling buffer and return
    (smoothed_dominant_emotion, smoothed_confidence_pct).

    If *new_scores* is None the last buffer entry is reused so the
    display never flickers to 'unknown' mid-stream.
    """
    if new_scores is None:
        if not buffer:
            return "unknown", 0.0
        new_scores = buffer[-1]

    buffer.append(new_scores)

    # Average each emotion score across the buffer
    avg = {lbl: sum(s.get(lbl, 0.0) for s in buffer) / len(buffer)
           for lbl in EMOTION_LABELS}

    dominant   = max(avg, key=avg.get)
    confidence = avg[dominant]
    return dominant, confidence


# 
# Session statistics tracker
# 
class SessionStats:
    """Accumulates per-emotion counts and confidence totals."""

    def __init__(self):
        self.counts     = defaultdict(int)
        self.conf_total = defaultdict(float)
        self.total      = 0

    def update(self, emotion: str, confidence: float):
        self.counts[emotion]     += 1
        self.conf_total[emotion] += confidence
        self.total               += 1

    def dominant(self):
        return max(self.counts, key=self.counts.get) if self.counts else "N/A"

    def distribution(self):
        """Return {emotion: percentage} rounded to one decimal place."""
        if not self.total:
            return {}
        return {e: round(100 * c / self.total, 1)
                for e, c in self.counts.items()}

    def avg_confidence(self, emotion: str) -> float:
        if self.counts[emotion] == 0:
            return 0.0
        return self.conf_total[emotion] / self.counts[emotion]

    def mental_health_score(self) -> float:
        """
        Weighted average of emotion valences, scaled to [0, 100].

        Score interpretation:
          ≥ 70  → Positive / good mood
          50–69 → Neutral / mixed
          < 50  → Predominantly negative affect
        """
        if not self.total:
            return 50.0

        dist  = self.distribution()
        raw   = sum(VALENCE.get(e, 0) * (pct / 100)
                    for e, pct in dist.items())
        # raw ∈ [−1, +1]  → rescale to [0, 100]
        return round((raw + 1) / 2 * 100, 1)


# 
# Drawing helpers
# 
def draw_overlay(frame, box, emotion: str, confidence: float, fps: float):
    """Render bounding box, emotion badge with confidence, and FPS."""
    x1, y1, x2, y2 = box
    font       = cv2.FONT_HERSHEY_SIMPLEX
    scale_lbl  = 0.72
    scale_fps  = 0.80
    thick      = 2

    # Bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), BOX_COLOR, 2)

    # Emotion + confidence badge above the box
    label = f"  {emotion.upper()}  {confidence:.1f}%  "
    (tw, th), base = cv2.getTextSize(label, font, scale_lbl, thick)
    by1 = max(0,               y1 - th - base - 8)
    by2 = y1
    bx2 = min(frame.shape[1],  x1 + tw + 4)

    cv2.rectangle(frame, (x1, by1), (bx2, by2), BADGE_COLOR, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - base - 2),
                font, scale_lbl, TEXT_COLOR, thick, cv2.LINE_AA)

    # FPS  (top-left)
    cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                font, scale_fps, FPS_COLOR, thick, cv2.LINE_AA)


def print_session_report(stats: SessionStats, csv_path: str):
    """Print a formatted end-of-session report to stdout."""
    sep = "═" * 52
    print(f"\n{sep}")
    print("  SESSION EMOTION REPORT")
    print(sep)
    print(f"  Total detections  : {stats.total}")
    print(f"  Dominant emotion  : {stats.dominant()}")
    print(f"\n  Emotion distribution:")
    for emo, pct in sorted(stats.distribution().items(),
                           key=lambda kv: -kv[1]):
        bar = "█" * int(pct / 2)
        print(f"    {emo:<10} {pct:5.1f}%  {bar}")
    score = stats.mental_health_score()
    mood  = ("Positive 😊" if score >= 70
             else "Neutral 😐" if score >= 50
             else "Negative 😟")
    print(f"\n  Mental-health score : {score} / 100  ({mood})")
    print(f"  Log saved to        : {csv_path}")
    print(sep + "\n")


# 
# Main

def main():
    #  MediaPipe setup
    mp_fd   = mp.solutions.face_detection
    mp_fm   = mp.solutions.face_mesh

    detector = mp_fd.FaceDetection(
        model_selection          = 0,             # short-range model (≤2 m)
        min_detection_confidence = MIN_DETECT_CONF,
    )
    face_mesh = mp_fm.FaceMesh(
        static_image_mode        = False,
        max_num_faces            = 1,
        refine_landmarks         = True,
        min_detection_confidence = MIN_DETECT_CONF,
        min_tracking_confidence  = MIN_TRACK_CONF,
    )

    #  CSV logger 
    csv_writer, csv_fh = init_csv(CSV_FILENAME)
    print(f"[INFO] Logging emotions to: {CSV_FILENAME}")

    #  Session state 
    stats          = SessionStats()
    emotion_buffer = deque(maxlen=SMOOTH_BUFFER_SIZE)

    frame_count   = 0
    last_emotion  = "unknown"
    last_conf     = 0.0
    last_scores   = None
    last_box      = None

    fps        = 0.0
    fps_timer  = time.time()
    fps_frames = 0

    #  Webcam 
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        return
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)   # minimise latency
    print("[INFO] Press 'q' to quit and view the session report.")

    #  Main loop 
    while True:
        ret, frame = cap.read()
        if not ret:
            print("[WARNING] Frame grab failed — retrying…")
            continue

        frame_count  += 1
        fps_frames   += 1

        # Update FPS every second
        elapsed = time.time() - fps_timer
        if elapsed >= 1.0:
            fps       = fps_frames / elapsed
            fps_timer = time.time()
            fps_frames = 0

        # Step 1 – Fast face detection
        coarse = detect_face(frame, detector)

        if coarse is not None:
            # Step 2 – Refine crop with FaceMesh landmarks
            refined = refine_crop_with_facemesh(frame, coarse, face_mesh)
            if refined is None:
                refined = coarse        # graceful fallback
            last_box = refined

            # Step 3 – DeepFace analysis (throttled to every FRAME_SKIP frames)
            if frame_count % FRAME_SKIP == 0:
                new_scores, raw_emotion, raw_conf = analyze_emotion(frame, refined)
                last_scores = new_scores            # may be None on failure

            # Step 4 – Temporal smoothing
            last_emotion, last_conf = smooth_emotion(emotion_buffer, last_scores)

            # Step 5 – Log + accumulate stats (once per analysis frame)
            if frame_count % FRAME_SKIP == 0 and last_emotion != "unknown":
                log_emotion(csv_writer, last_emotion, last_conf)
                stats.update(last_emotion, last_conf)

        # Step 6 – Render
        if last_box is not None and last_emotion != "unknown":
            draw_overlay(frame, last_box, last_emotion, last_conf, fps)
        else:
            cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, FPS_COLOR, 2, cv2.LINE_AA)
            cv2.putText(frame, "No face detected", (12, 68),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (80, 80, 255), 2, cv2.LINE_AA)

        cv2.imshow("Emotion Detection  [q = quit]", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    #  Cleanup 
    cap.release()
    detector.close()
    face_mesh.close()
    cv2.destroyAllWindows()
    csv_fh.close()

    #  Session report 
    print_session_report(stats, CSV_FILENAME)


if __name__ == "__main__":
    main()

[INFO] Logging emotions to: emotion_logs\session_20260305_130013.csv
[INFO] Press 'q' to quit and view the session report.

════════════════════════════════════════════════════
  SESSION EMOTION REPORT
════════════════════════════════════════════════════
  Total detections  : 117
  Dominant emotion  : neutral

  Emotion distribution:
    neutral     48.7%  ████████████████████████
    happy       29.9%  ██████████████
    sad         19.7%  █████████
    angry        1.7%  

  Mental-health score : 65.7 / 100  (Neutral 😐)
  Log saved to        : emotion_logs\session_20260305_130013.csv
════════════════════════════════════════════════════



In [ ]:
# CODE 15
"""

Emotion aggregator preference 2

Real-Time Facial Emotion Detection System — v3 (Improved Accuracy)
===================================================================
Key accuracy improvements over v2:
  • Minority-emotion amplification via SENSITIVITY_BOOST weights
  • Shorter smooth buffer (5 frames) so transient emotions aren't washed out
  • Frame skip reduced to 3 for faster negative-emotion capture
  • Dominant emotion chosen from BOOSTED scores, not DeepFace's raw pick
  • Confidence threshold lowered to 8 % so weak signals surface
  • CLAHE pre-processing sharpens facial contrast before DeepFace sees the crop
  • Larger crop padding (30 %) captures more context (brow furrow, jaw tension)

Pipeline : Webcam → MediaPipe FaceDetection → FaceMesh crop →
           CLAHE enhancement → DeepFace (every 3 frames) →
           Boost → Temporal smoothing → Display
Extras   : CSV logging  |  Session statistics  |  Mental-health score
"""

import csv
import os
import time
from collections import deque, defaultdict
from datetime import datetime

import cv2
import mediapipe as mp
import numpy as np
from deepface import DeepFace



# Configuration
FRAME_SKIP         = 3       # ↓ from 5 — catch fleeting negative emotions faster
SMOOTH_BUFFER_SIZE = 5       # ↓ from 8 — less history so transients aren't buried
CROP_PADDING       = 0.30    # ↑ from 0.20 — include brow/jaw context
FACE_CROP_SIZE     = 224
MIN_DETECT_CONF    = 0.55    # slightly more permissive detection
MIN_TRACK_CONF     = 0.55

LOG_DIR      = "emotion_logs"
CSV_FILENAME = os.path.join(
    LOG_DIR, f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
)

EMOTION_LABELS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]

# ── Sensitivity boost weights 
# DeepFace's FER model is heavily biased toward happy/neutral.
# Multiply raw probabilities by these factors BEFORE picking dominant.
# Negative-affect emotions are amplified; over-represented ones are dampened.
SENSITIVITY_BOOST = {
    "angry":    2.0,
    "disgust":  2.8,   # most under-detected — highest boost
    "fear":     2.5,
    "sad":      2.2,
    "surprise": 1.5,
    "neutral":  0.7,   # dampen the default catch-all
    "happy":    0.8,
}

# Minimum boosted probability (%) for an emotion to win as dominant
MIN_CONFIDENCE_PCT = 8.0

# Valence weights for mental-health score
VALENCE = {
    "happy":    1.0,
    "neutral":  0.3,
    "surprise": 0.1,
    "sad":     -0.6,
    "angry":   -0.8,
    "fear":    -0.7,
    "disgust": -0.5,
}

# ── Display colours (BGR) 
BOX_COLOR   = (0, 220, 110)
TEXT_COLOR  = (255, 255, 255)
BADGE_COLOR = (0, 160,  80)
FPS_COLOR   = (0, 200, 255)


# 
# CLAHE contrast enhancement
# 
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))

def enhance_crop(bgr_crop: np.ndarray) -> np.ndarray:
    """
    Apply CLAHE to the luminance channel so subtle muscle movements
    (brow furrow = fear/anger/disgust, lip corners = sad) become
    more visible to the FER model.
    """
    lab          = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2LAB)
    l, a, b      = cv2.split(lab)
    l_eq         = _clahe.apply(l)
    enhanced_lab = cv2.merge([l_eq, a, b])
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)


# 
# CSV logging
# 
def init_csv(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fh     = open(path, "a", newline="", encoding="utf-8")
    writer = csv.writer(fh)
    if os.path.getsize(path) == 0:
        writer.writerow(["timestamp", "emotion", "confidence_pct"])
    return writer, fh


def log_emotion(writer, emotion: str, confidence: float):
    writer.writerow([
        datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        emotion,
        f"{confidence:.2f}",
    ])


# 
# Face detection  (MediaPipe FaceDetection)
# 
def detect_face(frame, detector):
    """
    Return (x1,y1,x2,y2) for the highest-confidence face, or None.
    """
    h, w    = frame.shape[:2]
    results = detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if not results.detections:
        return None

    best = max(results.detections, key=lambda d: d.score[0])
    bb   = best.location_data.relative_bounding_box

    x1 = max(0,     int(bb.xmin * w))
    y1 = max(0,     int(bb.ymin * h))
    x2 = min(w - 1, int((bb.xmin + bb.width)  * w))
    y2 = min(h - 1, int((bb.ymin + bb.height) * h))

    return (x1, y1, x2, y2) if x2 > x1 and y2 > y1 else None


 
# Landmark-based crop refinement  (MediaPipe FaceMesh)

def refine_crop_with_facemesh(frame, coarse_box, face_mesh):
    """
    Tighten the bounding box to actual landmark extents, then pad.
    Falls back to coarse_box if mesh detection fails.
    """
    h, w    = frame.shape[:2]
    cx1, cy1, cx2, cy2 = coarse_box

    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None

    ch, cw  = crop.shape[:2]
    results = face_mesh.process(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

    if results.multi_face_landmarks:
        pts = results.multi_face_landmarks[0].landmark
        xs  = [p.x * cw for p in pts]
        ys  = [p.y * ch for p in pts]
        lx1 = int(min(xs)) + cx1;  ly1 = int(min(ys)) + cy1
        lx2 = int(max(xs)) + cx1;  ly2 = int(max(ys)) + cy1
    else:
        lx1, ly1, lx2, ly2 = cx1, cy1, cx2, cy2

    px  = int((lx2 - lx1) * CROP_PADDING)
    py  = int((ly2 - ly1) * CROP_PADDING)

    rx1 = max(0,     lx1 - px);  ry1 = max(0,     ly1 - py)
    rx2 = min(w - 1, lx2 + px);  ry2 = min(h - 1, ly2 + py)

    return (rx1, ry1, rx2, ry2) if rx2 > rx1 and ry2 > ry1 else None



# DeepFace emotion analysis  (with CLAHE + sensitivity boost)

def analyze_emotion(frame, box):
    """
    1. Crop & resize to FACE_CROP_SIZE.
    2. Apply CLAHE contrast enhancement.
    3. Run DeepFace.
    4. Apply SENSITIVITY_BOOST to raw probabilities.
    5. Re-normalise to 0–100 and pick the dominant emotion.

    Returns (boosted_scores, dominant_emotion, confidence_pct)
            or (None, None, None) on failure.
    """
    x1, y1, x2, y2 = box
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None, None, None

    resized  = cv2.resize(crop, (FACE_CROP_SIZE, FACE_CROP_SIZE))
    enhanced = enhance_crop(resized)          # CLAHE

    try:
        result = DeepFace.analyze(
            img_path          = enhanced,
            actions           = ["emotion"],
            enforce_detection = False,
            silent            = True,
        )
        if isinstance(result, list):
            result = result[0]

        raw_scores = result.get("emotion", {})

    except Exception:
        return None, None, None

    #  Apply sensitivity boost 
    boosted = {
        lbl: raw_scores.get(lbl, 0.0) * SENSITIVITY_BOOST.get(lbl, 1.0)
        for lbl in EMOTION_LABELS
    }

    # Re-normalise so scores still sum to 100
    total = sum(boosted.values()) or 1.0
    boosted = {lbl: (v / total) * 100.0 for lbl, v in boosted.items()}

    dominant   = max(boosted, key=boosted.get)
    confidence = boosted[dominant]

    # Reject if nothing clears the minimum threshold
    if confidence < MIN_CONFIDENCE_PCT:
        return None, None, None

    return boosted, dominant, confidence


 
# Temporal smoothing

def smooth_emotion(buffer: deque, new_scores: dict):
    """
    Average boosted scores over the rolling buffer.
    Reuses last entry if new_scores is None (face briefly missing).
    Returns (dominant_emotion, confidence_pct).
    """
    if new_scores is None:
        if not buffer:
            return "unknown", 0.0
        new_scores = buffer[-1]

    buffer.append(new_scores)

    avg = {
        lbl: sum(s.get(lbl, 0.0) for s in buffer) / len(buffer)
        for lbl in EMOTION_LABELS
    }

    dominant   = max(avg, key=avg.get)
    confidence = avg[dominant]
    return dominant, confidence



# Session statistics
 
class SessionStats:
    def __init__(self):
        self.counts     = defaultdict(int)
        self.conf_total = defaultdict(float)
        self.total      = 0

    def update(self, emotion: str, confidence: float):
        self.counts[emotion]     += 1
        self.conf_total[emotion] += confidence
        self.total               += 1

    def dominant(self):
        return max(self.counts, key=self.counts.get) if self.counts else "N/A"

    def distribution(self):
        if not self.total:
            return {}
        return {e: round(100 * c / self.total, 1) for e, c in self.counts.items()}

    def mental_health_score(self) -> float:
        """Valence-weighted score scaled to [0, 100]."""
        if not self.total:
            return 50.0
        dist = self.distribution()
        raw  = sum(VALENCE.get(e, 0) * (pct / 100) for e, pct in dist.items())
        return round((raw + 1) / 2 * 100, 1)



# Drawing
 
def draw_overlay(frame, box, emotion: str, confidence: float, fps: float):
    x1, y1, x2, y2 = box
    font  = cv2.FONT_HERSHEY_SIMPLEX
    thick = 2

    cv2.rectangle(frame, (x1, y1), (x2, y2), BOX_COLOR, 2)

    label           = f"  {emotion.upper()}  {confidence:.1f}%  "
    (tw, th), base  = cv2.getTextSize(label, font, 0.72, thick)
    by1 = max(0,              y1 - th - base - 8)
    bx2 = min(frame.shape[1], x1 + tw + 4)

    cv2.rectangle(frame, (x1, by1), (bx2, y1), BADGE_COLOR, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - base - 2),
                font, 0.72, TEXT_COLOR, thick, cv2.LINE_AA)

    cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                font, 0.80, FPS_COLOR, thick, cv2.LINE_AA)


def print_session_report(stats: SessionStats, csv_path: str):
    sep = "═" * 52
    print(f"\n{sep}\n  SESSION EMOTION REPORT\n{sep}")
    print(f"  Total detections  : {stats.total}")
    print(f"  Dominant emotion  : {stats.dominant()}")
    print(f"\n  Emotion distribution:")
    for emo, pct in sorted(stats.distribution().items(), key=lambda kv: -kv[1]):
        bar = "█" * int(pct / 2)
        print(f"    {emo:<10} {pct:5.1f}%  {bar}")
    score = stats.mental_health_score()
    mood  = ("Positive 😊" if score >= 70 else
             "Neutral  😐" if score >= 50 else "Negative 😟")
    print(f"\n  Mental-health score : {score} / 100  ({mood})")
    print(f"  Log saved to        : {csv_path}")
    print(sep + "\n")



# Main
 
def main():
    #  MediaPipe 
    detector  = mp.solutions.face_detection.FaceDetection(
        model_selection=0, min_detection_confidence=MIN_DETECT_CONF
    )
    face_mesh = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=False, max_num_faces=1, refine_landmarks=True,
        min_detection_confidence=MIN_DETECT_CONF,
        min_tracking_confidence=MIN_TRACK_CONF,
    )

    #  CSV 
    csv_writer, csv_fh = init_csv(CSV_FILENAME)
    print(f"[INFO] Logging to: {CSV_FILENAME}")
    print("[INFO] Press 'q' to quit.")

    #  State 
    stats          = SessionStats()
    emotion_buffer = deque(maxlen=SMOOTH_BUFFER_SIZE)

    frame_count  = 0
    last_emotion = "unknown"
    last_conf    = 0.0
    last_scores  = None
    last_box     = None

    fps = fps_frames = 0.0
    fps_timer = time.time()

    #  Webcam 
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        return
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        frame_count += 1
        fps_frames  += 1

        elapsed = time.time() - fps_timer
        if elapsed >= 1.0:
            fps = fps_frames / elapsed
            fps_timer  = time.time()
            fps_frames = 0

        # Step 1 – Detect face
        coarse = detect_face(frame, detector)

        if coarse is not None:
            # Step 2 – Refine crop
            refined  = refine_crop_with_facemesh(frame, coarse, face_mesh) or coarse
            last_box = refined

            # Step 3 – Analyse (throttled)
            if frame_count % FRAME_SKIP == 0:
                new_scores, _, _ = analyze_emotion(frame, refined)
                last_scores = new_scores

            # Step 4 – Smooth
            last_emotion, last_conf = smooth_emotion(emotion_buffer, last_scores)

            # Step 5 – Log & stats
            if frame_count % FRAME_SKIP == 0 and last_emotion != "unknown":
                log_emotion(csv_writer, last_emotion, last_conf)
                stats.update(last_emotion, last_conf)

        # Step 6 – Render
        if last_box is not None and last_emotion != "unknown":
            draw_overlay(frame, last_box, last_emotion, last_conf, fps)
        else:
            cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, FPS_COLOR, 2, cv2.LINE_AA)
            cv2.putText(frame, "No face detected", (12, 68),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (80, 80, 255), 2, cv2.LINE_AA)

        cv2.imshow("Emotion Detection v3  [q = quit]", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    #  Cleanup 
    cap.release()
    detector.close()
    face_mesh.close()
    cv2.destroyAllWindows()
    csv_fh.close()
    print_session_report(stats, CSV_FILENAME)


if __name__ == "__main__":
    main()

[INFO] Logging to: emotion_logs\session_20260305_130133.csv
[INFO] Press 'q' to quit.

════════════════════════════════════════════════════
  SESSION EMOTION REPORT
════════════════════════════════════════════════════
  Total detections  : 24
  Dominant emotion  : sad

  Emotion distribution:
    sad         87.5%  ███████████████████████████████████████████
    angry        8.3%  ████
    neutral      4.2%  ██

  Mental-health score : 21.1 / 100  (Negative 😟)
  Log saved to        : emotion_logs\session_20260305_130133.csv
════════════════════════════════════════════════════



In [ ]:
# CODE 16
"""
Emotion aggregator preference 1

Real-Time Facial Emotion Detection System — v4 (Advanced Accuracy)
===================================================================
Accuracy upgrades over v3:
  1. MULTI-SCALE ENSEMBLE  — DeepFace runs on 3 crop sizes; results are
     averaged so the model sees both fine detail and wider context.
  2. ADAPTIVE PRE-PROCESSING — bilateral filter (edge-preserving denoise) +
     adaptive gamma correction (normalises dark / over-exposed faces) +
     CLAHE contrast enhancement.
  3. LANDMARK GEOMETRY FEATURES — brow raise (fear/surprise), brow furrow
     (anger/disgust), mouth curvature (sad/happy), eye openness (fear)
     computed from FaceMesh points and blended into the final scores.
  4. EMOTION TRANSITION MATRIX — implausible jumps (e.g. happy → disgust)
     are penalised; plausible sequences (sad → fear) are preserved.
  5. EXPONENTIAL-DECAY SMOOTHING — recent frames count more than older
     ones, so the buffer reacts faster to genuine emotion changes.
  6. PER-LABEL CALIBRATION FLOOR — disgust / fear / sad cannot be zeroed
     out even if raw DeepFace probability is extremely low.
  7. SENSITIVITY BOOST (refined from v3) with tighter neutral/happy damping.

Pipeline:
  Webcam → FaceDetection → FaceMesh crop + geometry →
  Adaptive pre-proc → 3× DeepFace ensemble →
  Geometry blend → Transition penalty → Exp-decay smooth →
  Display + CSV + Session report
"""

import csv
import math
import os
import time
from collections import deque, defaultdict
from datetime import datetime

import cv2
import mediapipe as mp
import numpy as np
from deepface import DeepFace


 
# Configuration

FRAME_SKIP         = 3
SMOOTH_BUFFER_SIZE = 6
BASE_PADDING       = 0.28        # primary crop padding
FACE_CROP_SIZE     = 224
MIN_DETECT_CONF    = 0.50
MIN_TRACK_CONF     = 0.50
MIN_CONF_PCT       = 6.0         # absolute floor before we trust a result

# Multi-scale crop padding variants (applied on top of refined landmark box)
CROP_SCALES = [0.20, 0.30, 0.42]   # tight / normal / wide

# Sensitivity boost (re-normalised after application)
SENSITIVITY_BOOST = {
    "angry":    1.8,
    "disgust":  3.2,
    "fear":     2.8,
    "sad":      2.4,
    "surprise": 1.4,
    "neutral":  0.60,
    "happy":    0.75,
}

# Calibration floors — even if model gives 0, these minimums are injected
# before boosting.  Prevents minority classes being completely zeroed out.
CALIBRATION_FLOOR = {
    "disgust": 1.5,
    "fear":    1.5,
    "sad":     2.0,
    "angry":   1.0,
    "surprise": 0.5,
    "happy":   0.0,
    "neutral": 0.0,
}

# Emotion transition plausibility matrix  [from][to] ∈ (0, 1]
# Values < 1 penalise implausible jumps.
TRANSITION = {
    "happy":    {"happy":1.0,"neutral":0.9,"surprise":0.7,"sad":0.4,
                 "angry":0.3,"fear":0.3,"disgust":0.2},
    "neutral":  {"neutral":1.0,"happy":0.9,"sad":0.8,"angry":0.7,
                 "surprise":0.7,"fear":0.6,"disgust":0.5},
    "sad":      {"sad":1.0,"neutral":0.8,"fear":0.7,"angry":0.6,
                 "disgust":0.5,"surprise":0.4,"happy":0.3},
    "angry":    {"angry":1.0,"disgust":0.9,"sad":0.6,"neutral":0.5,
                 "fear":0.5,"surprise":0.4,"happy":0.2},
    "fear":     {"fear":1.0,"surprise":0.8,"sad":0.7,"neutral":0.6,
                 "angry":0.5,"disgust":0.4,"happy":0.2},
    "disgust":  {"disgust":1.0,"angry":0.8,"neutral":0.5,"sad":0.5,
                 "fear":0.4,"surprise":0.3,"happy":0.2},
    "surprise": {"surprise":1.0,"happy":0.8,"fear":0.7,"neutral":0.7,
                 "sad":0.4,"angry":0.4,"disgust":0.3},
    "unknown":  {e: 1.0 for e in
                 ["angry","disgust","fear","happy","sad","surprise","neutral"]},
}

# Geometry blending weight (0 = ignore geometry, 1 = geometry only)
GEOMETRY_BLEND = 0.22

# Exponential decay base for smoothing buffer (higher = more weight on recent)
EXP_DECAY = 0.72

EMOTION_LABELS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]

# Valence for mental-health score
VALENCE = {
    "happy":1.0,"neutral":0.3,"surprise":0.1,
    "sad":-0.6,"angry":-0.8,"fear":-0.7,"disgust":-0.5,
}

LOG_DIR      = "emotion_logs"
CSV_FILENAME = os.path.join(
    LOG_DIR, f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
)

# Display
BOX_COLOR   = (0, 220, 110)
TEXT_COLOR  = (255, 255, 255)
BADGE_COLOR = (30, 30, 30)
FPS_COLOR   = (0, 200, 255)
BAR_COLORS  = {          # per-emotion bar colours for the live chart
    "happy":   (0, 215, 255),
    "neutral": (180,180,180),
    "sad":     (200, 120,  40),
    "angry":   (50,  50, 220),
    "fear":    (130,  0, 200),
    "disgust": (40, 160,  40),
    "surprise":(0, 200, 200),
}

_clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(4, 4))


 
# Pre-processing

def adaptive_gamma(gray: np.ndarray) -> np.ndarray:
    """Auto-gamma: bright faces get γ>1 (darken), dim faces get γ<1 (brighten)."""
    mean = np.mean(gray) / 255.0
    gamma = math.log(0.5) / (math.log(mean) if mean > 0 else 1e-6)
    gamma = float(np.clip(gamma, 0.4, 2.5))
    table = np.array([(i / 255.0) ** gamma * 255 for i in range(256)],
                     dtype=np.uint8)
    return cv2.LUT(gray, table)


def preprocess_crop(bgr: np.ndarray) -> np.ndarray:
    """
    Bilateral denoise → adaptive gamma → CLAHE.
    Preserves edges (important for wrinkle / fold cues of disgust / anger).
    """
    # 1. Edge-preserving denoise
    denoised = cv2.bilateralFilter(bgr, d=7, sigmaColor=55, sigmaSpace=55)

    # 2. Adaptive gamma on luminance
    lab          = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b      = cv2.split(lab)
    l_gamma      = adaptive_gamma(l)

    # 3. CLAHE on gamma-corrected luminance
    l_clahe      = _clahe.apply(l_gamma)
    enhanced_lab = cv2.merge([l_clahe, a, b])
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)


 
# CSV

def init_csv(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fh = open(path, "a", newline="", encoding="utf-8")
    w  = csv.writer(fh)
    if os.path.getsize(path) == 0:
        w.writerow(["timestamp", "emotion", "confidence_pct"])
    return w, fh


def log_emotion(writer, emotion: str, confidence: float):
    writer.writerow([
        datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        emotion, f"{confidence:.2f}",
    ])



# Face detection
 
def detect_face(frame, detector):
    h, w    = frame.shape[:2]
    results = detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if not results.detections:
        return None
    best = max(results.detections, key=lambda d: d.score[0])
    bb   = best.location_data.relative_bounding_box
    x1 = max(0,     int(bb.xmin * w))
    y1 = max(0,     int(bb.ymin * h))
    x2 = min(w - 1, int((bb.xmin + bb.width)  * w))
    y2 = min(h - 1, int((bb.ymin + bb.height) * h))
    return (x1, y1, x2, y2) if x2 > x1 and y2 > y1 else None


 
# FaceMesh crop refinement + landmark geometry


# Landmark index groups (MediaPipe 468-point model)
_LM = {
    "left_brow":    [70, 63, 105, 66, 107],
    "right_brow":   [336,296,334,293,300],
    "left_eye":     [33, 160,158,133,153,144],
    "right_eye":    [362,385,387,263,373,380],
    "nose_tip":     [1],
    "mouth_left":   [61],
    "mouth_right":  [291],
    "mouth_top":    [13],
    "mouth_bottom": [14],
    "chin":         [152],
    "forehead":     [10],
}


def _lm_mean(landmarks, indices, w, h):
    """Average (x,y) pixel position for a group of landmark indices."""
    pts = [(landmarks[i].x * w, landmarks[i].y * h) for i in indices]
    return np.mean(pts, axis=0)


def extract_geometry_scores(landmarks, w, h) -> dict:
    """
    Compute normalised geometry signals and map them to emotion-score
    adjustments in [0, 100] range so they can be blended with DeepFace output.

    Signals:
      • brow_raise      → fear / surprise
      • brow_furrow     → anger / disgust
      • mouth_curve     → happy (up) / sad (down)
      • mouth_open      → surprise / fear
      • eye_openness    → fear / surprise
    """
    lm  = landmarks

    # Reference: inter-eye distance for normalisation
    le = _lm_mean(lm, _LM["left_eye"],  w, h)
    re = _lm_mean(lm, _LM["right_eye"], w, h)
    inter_eye = max(np.linalg.norm(le - re), 1.0)

    nose  = _lm_mean(lm, _LM["nose_tip"],    w, h)
    fhead = _lm_mean(lm, _LM["forehead"],    w, h)
    chin  = _lm_mean(lm, _LM["chin"],        w, h)
    face_h = max(np.linalg.norm(fhead - chin), 1.0)

    # Brow positions (y; lower value = higher on screen = raised)
    lb = _lm_mean(lm, _LM["left_brow"],  w, h)
    rb = _lm_mean(lm, _LM["right_brow"], w, h)
    brow_y_norm = ((le[1] - lb[1]) + (re[1] - rb[1])) / (2 * inter_eye)

    # Brow x-spread (furrow = brows closer together)
    brow_x_spread = abs(lb[0] - rb[0]) / inter_eye

    # Mouth curvature: corners vs centre top/bottom
    ml  = _lm_mean(lm, _LM["mouth_left"],   w, h)
    mr  = _lm_mean(lm, _LM["mouth_right"],  w, h)
    mt  = _lm_mean(lm, _LM["mouth_top"],    w, h)
    mb  = _lm_mean(lm, _LM["mouth_bottom"], w, h)
    mouth_mid_y  = (ml[1] + mr[1]) / 2
    mouth_curve  = (mouth_mid_y - mt[1]) / inter_eye  # + = corners up = happy

    # Mouth openness
    mouth_open = np.linalg.norm(mt - mb) / inter_eye

    # Eye openness (average vertical span)
    def eye_open(indices):
        pts = np.array([(lm[i].x * w, lm[i].y * h) for i in indices])
        return (np.max(pts[:,1]) - np.min(pts[:,1])) / inter_eye

    eye_openness = (eye_open(_LM["left_eye"]) + eye_open(_LM["right_eye"])) / 2

    #  Map to emotion score adjustments 
    g = {e: 0.0 for e in EMOTION_LABELS}

    # Brow raise  → fear / surprise
    raise_sig = float(np.clip(brow_y_norm, 0, 1))
    g["fear"]     += raise_sig * 40
    g["surprise"] += raise_sig * 35

    # Brow furrow (narrow spread) → anger / disgust
    furrow_sig = float(np.clip(2.0 - brow_x_spread, 0, 1))
    g["angry"]   += furrow_sig * 35
    g["disgust"] += furrow_sig * 30

    # Mouth curve up → happy ; down → sad
    if mouth_curve > 0.05:
        g["happy"] += float(np.clip(mouth_curve * 80, 0, 50))
    else:
        g["sad"]   += float(np.clip(-mouth_curve * 100, 0, 50))

    # Mouth open → surprise / fear
    open_sig = float(np.clip((mouth_open - 0.15) * 80, 0, 40))
    g["surprise"] += open_sig * 0.6
    g["fear"]     += open_sig * 0.4

    # Wide eyes → fear / surprise
    eye_sig = float(np.clip((eye_openness - 0.2) * 60, 0, 35))
    g["fear"]     += eye_sig * 0.55
    g["surprise"] += eye_sig * 0.45

    # Normalise to 0–100
    total = sum(g.values()) or 1.0
    return {k: (v / total) * 100 for k, v in g.items()}


def refine_crop_with_facemesh(frame, coarse_box, face_mesh, padding):
    """Landmark-tightened crop with variable padding. Returns box + landmarks."""
    h, w = frame.shape[:2]
    cx1, cy1, cx2, cy2 = coarse_box

    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None, None

    ch, cw  = crop.shape[:2]
    results = face_mesh.process(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

    landmarks = None
    if results.multi_face_landmarks:
        pts = results.multi_face_landmarks[0].landmark
        landmarks = pts          # keep for geometry
        xs  = [p.x * cw for p in pts]
        ys  = [p.y * ch for p in pts]
        lx1 = int(min(xs)) + cx1;  ly1 = int(min(ys)) + cy1
        lx2 = int(max(xs)) + cx1;  ly2 = int(max(ys)) + cy1
    else:
        lx1, ly1, lx2, ly2 = cx1, cy1, cx2, cy2

    px  = int((lx2 - lx1) * padding)
    py  = int((ly2 - ly1) * padding)
    rx1 = max(0,     lx1 - px);  ry1 = max(0,     ly1 - py)
    rx2 = min(w - 1, lx2 + px);  ry2 = min(h - 1, ly2 + py)

    if rx2 <= rx1 or ry2 <= ry1:
        return None, None

    return (rx1, ry1, rx2, ry2), landmarks



# DeepFace — single-scale analysis
 
def _deepface_scores(bgr_224: np.ndarray) -> dict | None:
    """
    Run DeepFace on a pre-processed 224×224 BGR crop.
    Returns raw probability dict or None on failure.
    """
    try:
        result = DeepFace.analyze(
            img_path=bgr_224, actions=["emotion"],
            enforce_detection=False, silent=True,
        )
        if isinstance(result, list):
            result = result[0]
        return result.get("emotion", {})
    except Exception:
        return None

 
# Multi-scale ensemble analysis

def analyze_emotion(frame, base_box, face_mesh):
    """
    1. Building CROP_SCALES number of crops at different padding levels.
    2. Pre-process each with bilateral + gamma + CLAHE.
    3. Run DeepFace on all; average valid results (ensemble).
    4. Applying calibration floors, sensitivity boost, re-normalise.
    5. Extracting landmark geometry scores and blend.
    6. Returns (final_scores, dominant, confidence).
    """
    h, w = frame.shape[:2]
    cx1, cy1, cx2, cy2 = base_box

    all_raw   = []
    landmarks = None

    for scale in CROP_SCALES:
        box, lm = refine_crop_with_facemesh(frame, base_box, face_mesh, scale)
        if box is None:
            continue
        if lm is not None and landmarks is None:
            landmarks = lm   # keep first successful landmark set

        x1, y1, x2, y2 = box
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        resized   = cv2.resize(crop, (FACE_CROP_SIZE, FACE_CROP_SIZE))
        processed = preprocess_crop(resized)
        scores    = _deepface_scores(processed)
        if scores is not None:
            all_raw.append(scores)

    if not all_raw:
        return None, None, None

    #  Ensemble average 
    ensemble = {
        lbl: sum(s.get(lbl, 0.0) for s in all_raw) / len(all_raw)
        for lbl in EMOTION_LABELS
    }

    # ── Calibration floors 
    for lbl, floor in CALIBRATION_FLOOR.items():
        ensemble[lbl] = max(ensemble[lbl], floor)

    # ── Sensitivity boost 
    boosted = {lbl: ensemble[lbl] * SENSITIVITY_BOOST.get(lbl, 1.0)
               for lbl in EMOTION_LABELS}
    total   = sum(boosted.values()) or 1.0
    boosted = {lbl: (v / total) * 100 for lbl, v in boosted.items()}

    # ── Geometry blend 
    if landmarks is not None:
        try:
            # landmarks are relative to the crop — convert using first scale box
            box0, _ = refine_crop_with_facemesh(
                frame, base_box, face_mesh, CROP_SCALES[1]
            )
            if box0:
                bx1, by1, bx2, by2 = box0
                bh = by2 - by1 or 1; bw = bx2 - bx1 or 1
                geo = extract_geometry_scores(landmarks, bw, bh)
                boosted = {
                    lbl: (1 - GEOMETRY_BLEND) * boosted[lbl]
                         + GEOMETRY_BLEND    * geo.get(lbl, 0.0)
                    for lbl in EMOTION_LABELS
                }
                total = sum(boosted.values()) or 1.0
                boosted = {lbl: (v / total) * 100 for lbl, v in boosted.items()}
        except Exception:
            pass   # geometry is supplementary — never crash on it

    dominant   = max(boosted, key=boosted.get)
    confidence = boosted[dominant]

    if confidence < MIN_CONF_PCT:
        return None, None, None

    return boosted, dominant, confidence



# Exponential-decay temporal smoothing + transition penalty

def smooth_emotion(buffer: deque, new_scores: dict, prev_emotion: str):
    """
    Weighted average across the buffer (recent frames count more),
    then apply transition plausibility penalty.
    Returns (dominant_emotion, confidence_pct).
    """
    if new_scores is None:
        if not buffer:
            return "unknown", 0.0
        new_scores = buffer[-1]

    buffer.append(new_scores)
    n = len(buffer)

    # Exponential decay weights: most recent = 1.0, oldest = EXP_DECAY^(n-1)
    weights = [EXP_DECAY ** (n - 1 - i) for i in range(n)]
    w_total = sum(weights)

    avg = {
        lbl: sum(weights[i] * buffer[i].get(lbl, 0.0) for i in range(n)) / w_total
        for lbl in EMOTION_LABELS
    }

    # Apply transition plausibility penalty
    trans_row = TRANSITION.get(prev_emotion, TRANSITION["unknown"])
    penalised = {lbl: avg[lbl] * trans_row.get(lbl, 0.8)
                 for lbl in EMOTION_LABELS}

    dominant   = max(penalised, key=penalised.get)
    total      = sum(penalised.values()) or 1.0
    confidence = (penalised[dominant] / total) * 100

    return dominant, confidence



# Session statistics

class SessionStats:
    def __init__(self):
        self.counts     = defaultdict(int)
        self.conf_total = defaultdict(float)
        self.total      = 0

    def update(self, emotion, confidence):
        self.counts[emotion]     += 1
        self.conf_total[emotion] += confidence
        self.total               += 1

    def dominant(self):
        return max(self.counts, key=self.counts.get) if self.counts else "N/A"

    def distribution(self):
        if not self.total: return {}
        return {e: round(100 * c / self.total, 1)
                for e, c in self.counts.items()}

    def mental_health_score(self) -> float:
        if not self.total: return 50.0
        dist = self.distribution()
        raw  = sum(VALENCE.get(e, 0) * (p / 100) for e, p in dist.items())
        return round((raw + 1) / 2 * 100, 1)


 
# Drawing

def draw_emotion_bars(frame, scores: dict, x_origin: int, y_origin: int):
    """Draw a compact probability bar chart in the corner of the frame."""
    bar_w_max = 120
    bar_h     = 14
    gap       = 5
    font      = cv2.FONT_HERSHEY_SIMPLEX

    for i, lbl in enumerate(EMOTION_LABELS):
        pct = scores.get(lbl, 0.0)
        bw  = int(bar_w_max * pct / 100)
        y   = y_origin + i * (bar_h + gap)

        # Background
        cv2.rectangle(frame, (x_origin, y),
                      (x_origin + bar_w_max, y + bar_h), (50, 50, 50), -1)
        # Fill
        if bw > 0:
            cv2.rectangle(frame, (x_origin, y),
                          (x_origin + bw, y + bar_h),
                          BAR_COLORS.get(lbl, (200, 200, 200)), -1)

        cv2.putText(frame, f"{lbl[:4]} {pct:4.1f}%",
                    (x_origin + bar_w_max + 4, y + bar_h - 2),
                    font, 0.38, (220, 220, 220), 1, cv2.LINE_AA)


def draw_overlay(frame, box, emotion, confidence, fps, scores):
    x1, y1, x2, y2 = box
    font  = cv2.FONT_HERSHEY_SIMPLEX
    thick = 2

    # Bounding box with rounded feel (just thicker corners)
    cv2.rectangle(frame, (x1, y1), (x2, y2), BOX_COLOR, 2)
    corner = 14
    for dx, dy in [(-1,-1),(1,-1),(-1,1),(1,1)]:
        px, py = (x1 if dx < 0 else x2), (y1 if dy < 0 else y2)
        cv2.line(frame, (px, py), (px + dx*corner, py), BOX_COLOR, 4)
        cv2.line(frame, (px, py), (px, py + dy*corner), BOX_COLOR, 4)

    # Badge
    label          = f"  {emotion.upper()}  {confidence:.1f}%  "
    (tw, th), base = cv2.getTextSize(label, font, 0.72, thick)
    by1 = max(0,              y1 - th - base - 8)
    bx2 = min(frame.shape[1], x1 + tw + 4)
    cv2.rectangle(frame, (x1, by1), (bx2, y1), BADGE_COLOR, -1)
    cv2.rectangle(frame, (x1, by1), (bx2, y1), BOX_COLOR,   1)
    cv2.putText(frame, label, (x1 + 2, y1 - base - 2),
                font, 0.72, TEXT_COLOR, thick, cv2.LINE_AA)

    # FPS
    cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                font, 0.80, FPS_COLOR, thick, cv2.LINE_AA)

    # Probability bars (bottom-left)
    if scores:
        h = frame.shape[0]
        draw_emotion_bars(frame, scores,
                          x_origin=10,
                          y_origin=h - len(EMOTION_LABELS) * 19 - 10)


def print_session_report(stats: SessionStats, csv_path: str):
    sep = "═" * 54
    print(f"\n{sep}\n  SESSION EMOTION REPORT  (v4 — Advanced)\n{sep}")
    print(f"  Total detections  : {stats.total}")
    print(f"  Dominant emotion  : {stats.dominant()}")
    print(f"\n  Emotion distribution:")
    for emo, pct in sorted(stats.distribution().items(), key=lambda kv: -kv[1]):
        bar = "█" * int(pct / 2)
        print(f"    {emo:<10} {pct:5.1f}%  {bar}")
    score = stats.mental_health_score()
    mood  = ("Positive 😊" if score >= 70 else
             "Neutral  😐" if score >= 50 else "Negative 😟")
    print(f"\n  Mental-health score : {score} / 100  ({mood})")
    print(f"  Log saved to        : {csv_path}")
    print(sep + "\n")


 
# Main

def main():
    # MediaPipe setup 
    detector = mp.solutions.face_detection.FaceDetection(
        model_selection=0, min_detection_confidence=MIN_DETECT_CONF
    )
    face_mesh = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=False, max_num_faces=1, refine_landmarks=True,
        min_detection_confidence=MIN_DETECT_CONF,
        min_tracking_confidence=MIN_TRACK_CONF,
    )

    #  CSV 
    csv_writer, csv_fh = init_csv(CSV_FILENAME)
    print(f"[INFO] Logging to : {CSV_FILENAME}")
    print("[INFO] Press 'q' to quit.\n")

    #  State 
    stats          = SessionStats()
    emotion_buffer = deque(maxlen=SMOOTH_BUFFER_SIZE)

    frame_count  = 0
    last_emotion = "unknown"
    last_conf    = 0.0
    last_scores  = None
    last_df_scores = None   # last valid DeepFace scores dict
    last_box     = None

    fps = fps_frames = 0.0
    fps_timer = time.time()

    # Webcam 
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam."); return
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        frame_count += 1
        fps_frames  += 1
        elapsed = time.time() - fps_timer
        if elapsed >= 1.0:
            fps       = fps_frames / elapsed
            fps_timer = time.time()
            fps_frames = 0

        # Step 1: Fast face detection
        coarse = detect_face(frame, detector)

        if coarse is not None:
            # Step 2: DeepFace ensemble (throttled)
            if frame_count % FRAME_SKIP == 0:
                new_scores, _, _ = analyze_emotion(frame, coarse, face_mesh)
                last_df_scores = new_scores

            # Derive bounding box for display (use normal padding)
            box, _ = refine_crop_with_facemesh(
                frame, coarse, face_mesh, BASE_PADDING
            )
            if box:
                last_box = box

            # Step 3: Smooth with transition penalty
            last_emotion, last_conf = smooth_emotion(
                emotion_buffer, last_df_scores, last_emotion
            )

            # Step 4: Log + stats
            if frame_count % FRAME_SKIP == 0 and last_emotion != "unknown":
                log_emotion(csv_writer, last_emotion, last_conf)
                stats.update(last_emotion, last_conf)

            # Use current buffer average for bar display
            if emotion_buffer:
                n = len(emotion_buffer)
                last_scores = {
                    lbl: sum(s.get(lbl, 0) for s in emotion_buffer) / n
                    for lbl in EMOTION_LABELS
                }

        # ── Step 5: Render ────────────────────────────────────────────
        if last_box is not None and last_emotion != "unknown":
            draw_overlay(frame, last_box, last_emotion,
                         last_conf, fps, last_scores)
        else:
            cv2.putText(frame, f"FPS: {fps:.1f}", (12, 32),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, FPS_COLOR, 2, cv2.LINE_AA)
            cv2.putText(frame, "No face detected", (12, 68),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (80, 80, 255), 2,
                        cv2.LINE_AA)

        cv2.imshow("Emotion Detection v4 — Advanced  [q = quit]", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # Cleanup 
    cap.release()
    detector.close()
    face_mesh.close()
    cv2.destroyAllWindows()
    csv_fh.close()
    print_session_report(stats, CSV_FILENAME)


if __name__ == "__main__":
    main()

[INFO] Logging to : emotion_logs\session_20260305_130159.csv
[INFO] Press 'q' to quit.


══════════════════════════════════════════════════════
  SESSION EMOTION REPORT  (v4 — Advanced)
══════════════════════════════════════════════════════
  Total detections  : 142
  Dominant emotion  : sad

  Emotion distribution:
    sad         49.3%  ████████████████████████
    angry       27.5%  █████████████
    neutral     16.2%  ████████
    fear         6.3%  ███
    surprise     0.7%  

  Mental-health score : 24.5 / 100  (Negative 😟)
  Log saved to        : emotion_logs\session_20260305_130159.csv
══════════════════════════════════════════════════════

